In [1]:
print("Kernel avviata con successo")

Kernel avviata con successo


In [2]:
# CELL 2: Visual Feature Extraction with CLIP
import os
import cv2
import torch
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
from transformers import CLIPProcessor, CLIPModel
import torch
print(torch.__version__) 

2.6.0+cu124


In [ ]:
#!pip uninstall torch torchvision torchaudio -y
#!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
#!pip install --upgrade torch torchvision torchaudio
#!pip install ipywidgets
#!pip install resampy

In [3]:
PROJECT_PATH = "Thesis_Data"

# 1. Load CLIP
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading CLIP model on {device}...")
model_clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor_clip = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def extract_clip_features(video_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0: fps = 30
    
    frame_features = []
    count = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        # Extract 1 frame per second
        if count % int(fps) == 0:
            # Convert BGR (OpenCV) to RGB (PIL)
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(img)
            
            inputs = processor_clip(images=pil_img, return_tensors="pt").to(device)
            
            with torch.no_grad():
                # --- THE BULLETPROOF FIX ---
                # 1. Run the vision part of the model explicitly (This outputs the "Object")
                vision_outputs = model_clip.vision_model(pixel_values=inputs['pixel_values'])
                
                # 2. Extract the raw tensor from inside that Object
                pooled_tensor = vision_outputs.pooler_output 
                
                # 3. Project it into the final 512-dimensional CLIP space
                features = model_clip.visual_projection(pooled_tensor)
                
            frame_features.append(features.cpu().numpy().flatten())
        count += 1
        
    cap.release()
    
    if not frame_features:
        return np.zeros(512 * 4)  # 4x perché usiamo 4 statistiche
    
    arr = np.array(frame_features)
    mean = np.mean(arr, axis=0)
    std  = np.std(arr, axis=0)
    mx   = np.max(arr, axis=0)
    mn   = np.min(arr, axis=0)
    
    return np.concatenate([mean, std, mx, mn])  # → 2048 dim
        
    ## Average the features across the whole video
    #return np.mean(frame_features, axis=0)

# 2. Process all videos
visual_features_dict_clip = {}
video_files = [f for f in os.listdir(f"{PROJECT_PATH}/videos") if f.endswith('.mp4')]

Loading CLIP model on cuda...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [4]:
# CELL 2: Visual Feature Extraction with CLIP



print("Extracting CLIP features (Visual)...")
for v_file in tqdm(video_files):
    v_id = v_file.replace(".mp4", "")
    v_path = f"{PROJECT_PATH}/videos/{v_file}"
    try:
        visual_features_dict_clip[v_id] = extract_clip_features(v_path)
    except Exception as e:
        print(f"Error on {v_id}: {e}")

# 3. Save with a NEW name
np.save(f"{PROJECT_PATH}/visual_features_clip.npy", visual_features_dict_clip)
print("CLIP Visual features saved successfully!")

Extracting CLIP features (Visual)...


  0%|          | 0/486 [00:00<?, ?it/s]

CLIP Visual features saved successfully!


In [5]:
# CELL 3: Audio Feature Extraction with VGGishm

# 1. Load VGGish from PyTorch Hub
print("Loading VGGish model...")
# We use harritaylor's implementation which is the standard PyTorch port for VGGish
vggish = torch.hub.load('harritaylor/torchvggish', 'vggish')
vggish.eval()

def extract_vggish_features(audio_path):
    # The harritaylor VGGish port takes a wav file path directly,
    # automatically resamples it to 16kHz, computes the mel-spectrogram, 
    # and runs it through the network!
    with torch.no_grad():
        # Outputs shape: [number_of_seconds, 128]
        embeddings = vggish.forward(audio_path)
    
    # Average the embeddings over time to get one 128-D vector for the whole ad
    # Convert tensor to numpy
    features_np = embeddings.cpu().numpy()
    
    if len(features_np) == 0:
        return np.zeros(128)
        
    return np.mean(features_np, axis=0)

# 2. Process all audio files
audio_features_dict_vggish = {}
audio_files = [f for f in os.listdir(f"{PROJECT_PATH}/audio") if f.endswith('.wav')]


Loading VGGish model...


Using cache found in C:\Users\alberto.ferrante_kin/.cache\torch\hub\harritaylor_torchvggish_master
c:\Users\alberto.ferrante_kin\Desktop\thesis-notebooks\.venv\Lib\site-packages\torch\serialization.py:1754: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  result = unpickler.load()


In [6]:

print("Extracting VGGish features (Audio)...")
for a_file in tqdm(audio_files):
    v_id = a_file.replace(".wav", "")
    a_path = f"{PROJECT_PATH}/audio/{a_file}"
    try:
        audio_features_dict_vggish[v_id] = extract_vggish_features(a_path)
    except Exception as e:
        print(f"Error on {v_id}: {e}")

# 3. Save with a NEW name
np.save(f"{PROJECT_PATH}/audio_features_vggish.npy", audio_features_dict_vggish)
print("VGGish Audio features saved successfully!")

Extracting VGGish features (Audio)...


  0%|          | 0/486 [00:00<?, ?it/s]

VGGish Audio features saved successfully!


In [7]:
# MASTER CELL: Data Loading, Deep Late Fusion, & Early Stopping
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score



In [8]:
# ==========================================
# 1. LOAD AND PREPARE DATA (Fixes the NameError)
# ==========================================
PROJECT_PATH = "Thesis_Data"
LABELS_PATH = "videos_with_sentiment_labels.csv"

df = pd.read_csv(LABELS_PATH)
visual_dict = np.load(f"{PROJECT_PATH}/visual_features_clip.npy", allow_pickle=True).item()
audio_dict = np.load(f"{PROJECT_PATH}/audio_features_vggish.npy", allow_pickle=True).item()

X_visual, X_audio, y_labels = [], [], []

for index, row in df.iterrows():
    v_id = row['video_id']
    label = row['majority_sentiment']
    if v_id in visual_dict and v_id in audio_dict:
        X_visual.append(visual_dict[v_id])
        X_audio.append(audio_dict[v_id])
        y_labels.append(label)

X_visual = np.array(X_visual)
X_audio = np.array(X_audio)
y_labels = np.array(y_labels)

le = LabelEncoder()
y_encoded = le.fit_transform(y_labels)



In [9]:
# ==========================================
# 2. DEFINE THE DEEP LATE FUSION NETWORK
# ==========================================
class DeepLateFusionMLP(nn.Module):
    def __init__(self, visual_dim=2048, audio_dim=512, num_classes=3):
        super().__init__()
        
        self.visual_net = nn.Sequential(
            nn.Linear(visual_dim, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(0.4), # Con batch_size=32 e 5-fold, i batch di training possono essere piccoli e BatchNorm1d diventa instabile. LayerNorm funziona indipendentemente dalla dimensione del batch ed è più stabile.
            nn.Linear(512, 256),        nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 128),        nn.GELU()
        )
        
        self.audio_net = nn.Sequential(
            nn.Linear(audio_dim, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 128),       nn.GELU()
        )
        
        # Fusion: 256 → classi
        self.classifier = nn.Sequential(
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(128, 64),  nn.GELU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, visual_x, audio_x):
        v_feats = self.visual_net(visual_x)
        a_feats = self.audio_net(audio_x)
        combined = torch.cat((v_feats, a_feats), dim=1) 
        return self.classifier(combined)

In [ ]:
# ==========================================
# 3. SETUP TRAINING & EARLY STOPPING
# ==========================================
weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_encoded), y=y_encoded)
class_weights_tensor = torch.tensor(weights, dtype=torch.float32)

k_folds = 5
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

all_true_labels = []
all_predictions = []
fold_accuracies = []

MAX_EPOCHS = 100
PATIENCE = 10  # Stop if validation loss doesn't improve for 10 epochs

print(f"\nStarting 5-Fold CV with Early Stopping (Patience: {PATIENCE})...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_visual, y_encoded)):
    print(f"\n--- FOLD {fold + 1}/{k_folds} ---")
    
    # Split and Scale Data
    X_v_train, X_v_val = X_visual[train_idx], X_visual[val_idx]
    X_a_train, X_a_val = X_audio[train_idx], X_audio[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]
    
    scaler_v = StandardScaler()
    X_v_train_scaled = scaler_v.fit_transform(X_v_train)
    X_v_val_scaled = scaler_v.transform(X_v_val)
    
    scaler_a = StandardScaler()
    X_a_train_scaled = scaler_a.fit_transform(X_a_train)
    X_a_val_scaled = scaler_a.transform(X_a_val)
    
    train_loader = DataLoader(TensorDataset(
        torch.tensor(X_v_train_scaled, dtype=torch.float32), 
        torch.tensor(X_a_train_scaled, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long)
    ), batch_size=32, shuffle=True)
    
    val_loader = DataLoader(TensorDataset(
        torch.tensor(X_v_val_scaled, dtype=torch.float32), 
        torch.tensor(X_a_val_scaled, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.long)
    ), batch_size=32, shuffle=False)
    
    model = DeepLateFusionMLP(visual_dim=X_visual.shape[1], audio_dim=X_audio.shape[1], num_classes=len(le.classes_))
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor) 
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau( # STUDIA
        optimizer, mode='min', factor=0.5, patience=5, verbose=True
    )
    best_val_loss = float('inf')
    epochs_no_improve = 0
    
    for epoch in range(MAX_EPOCHS):
        # -- TRAINING PHASE --
        model.train()
        for inputs_v, inputs_a, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs_v, inputs_a) 
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
        # -- VALIDATION PHASE (For Early Stopping) --
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs_v, inputs_a, labels in val_loader:
                outputs = model(inputs_v, inputs_a)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
        val_loss /= len(val_loader)
        scheduler.step(val_loss)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            best_model_state = model.state_dict()  # ← SALVA IL MIGLIOR STATO
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping triggered at Epoch {epoch + 1}! (Best Val Loss: {best_val_loss:.4f})")
            break
            

    # RIPRISTINA il miglior modello prima della valutazione
    model.load_state_dict(best_model_state)        
    # -- EVALUATE FOLD --
    model.eval()
    fold_preds, fold_labels = [], []
    with torch.no_grad():
        for inputs_v, inputs_a, labels in val_loader:
            outputs = model(inputs_v, inputs_a)
            _, preds = torch.max(outputs, 1)
            fold_preds.extend(preds.numpy())
            fold_labels.extend(labels.numpy())
            
    fold_acc = accuracy_score(fold_labels, fold_preds)
    fold_accuracies.append(fold_acc)
    print(f"Fold {fold + 1} Final Accuracy: {fold_acc:.4f}")
    
    all_true_labels.extend(fold_labels)
    all_predictions.extend(fold_preds)

print("\n" + "="*45)
print("=== FINAL DEEP LATE FUSION RESULTS ===")
print("="*45)
print(f"Average Accuracy: {np.mean(fold_accuracies):.4f} (+/- {np.std(fold_accuracies):.4f})\n")
print("Detailed Classification Report:")
print(classification_report(all_true_labels, all_predictions, target_names=le.classes_))


Starting 5-Fold CV with Early Stopping (Patience: 10)...

--- FOLD 1/5 ---


c:\Users\alberto.ferrante_kin\Desktop\thesis-notebooks\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Early stopping triggered at Epoch 11! (Best Val Loss: 1.1018)
Fold 1 Final Accuracy: 0.6222

--- FOLD 2/5 ---


c:\Users\alberto.ferrante_kin\Desktop\thesis-notebooks\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Early stopping triggered at Epoch 12! (Best Val Loss: 1.0337)
Fold 2 Final Accuracy: 0.5281

--- FOLD 3/5 ---


...

Early stopping triggered at Epoch 12! (Best Val Loss: 1.0856)
Fold 3 Final Accuracy: 0.5393

--- FOLD 4/5 ---


c:\Users\alberto.ferrante_kin\Desktop\thesis-notebooks\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Early stopping triggered at Epoch 12! (Best Val Loss: 1.0857)
Fold 4 Final Accuracy: 0.5056

--- FOLD 5/5 ---


c:\Users\alberto.ferrante_kin\Desktop\thesis-notebooks\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Early stopping triggered at Epoch 11! (Best Val Loss: 1.0962)
Fold 5 Final Accuracy: 0.5618

=== FINAL DEEP LATE FUSION RESULTS ===
Average Accuracy: 0.5514 (+/- 0.0398)

Detailed Classification Report:
              precision    recall  f1-score   support

    Negative       0.18      0.13      0.15        52
     Neutral       0.42      0.38      0.40       136
    Positive       0.66      0.72      0.69       258

    accuracy                           0.55       446
   macro avg       0.42      0.41      0.41       446
weighted avg       0.53      0.55      0.54       446



In [ ]:
# MASTER CELL: Binary Classification, Deep Late Fusion, & Early Stopping
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score

# ==========================================
# 1. LOAD AND PREPARE DATA (BINARY)
# ==========================================
PROJECT_PATH = "Thesis_Data"
LABELS_PATH = "videos_with_sentiment_labels.csv"

df = pd.read_csv(LABELS_PATH)

# --- THE BINARY FIX ---
# Combine Neutral and Negative into 'Non-Positive'
df['binary_sentiment'] = df['majority_sentiment'].apply(
    lambda x: 'Positive' if x == 'Positive' else 'Non-Positive'
)

visual_dict = np.load(f"{PROJECT_PATH}/visual_features_clip.npy", allow_pickle=True).item()
audio_dict = np.load(f"{PROJECT_PATH}/audio_features_vggish.npy", allow_pickle=True).item()

X_visual, X_audio, y_labels = [], [], []

for index, row in df.iterrows():
    v_id = row['video_id']
    label = row['binary_sentiment'] # Use the new binary label
    
    if v_id in visual_dict and v_id in audio_dict:
        X_visual.append(visual_dict[v_id])
        X_audio.append(audio_dict[v_id])
        y_labels.append(label)

X_visual = np.array(X_visual)
X_audio = np.array(X_audio)
y_labels = np.array(y_labels)

le = LabelEncoder()
y_encoded = le.fit_transform(y_labels)

print(f"Data Loaded! Binary Distribution:")
unique, counts = np.unique(y_encoded, return_counts=True)
for i in range(len(unique)):
    print(f"{le.classes_[i]}: {counts[i]} videos")

# ==========================================
# 2. DEFINE THE DEEP LATE FUSION NETWORK
# ==========================================
class DeepLateFusionMLP(nn.Module):
    def __init__(self, visual_dim=2048, audio_dim=512, num_classes=3):
        super().__init__()
        
        self.visual_net = nn.Sequential(
            nn.Linear(visual_dim, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(512, 256),        nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 128),        nn.GELU()
        )
        
        self.audio_net = nn.Sequential(
            nn.Linear(audio_dim, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 128),       nn.GELU()
        )
        
        # Fusion: 256 → classi
        self.classifier = nn.Sequential(
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(128, 64),  nn.GELU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, visual_x, audio_x):
        v_feats = self.visual_net(visual_x)
        a_feats = self.audio_net(audio_x)
        combined = torch.cat((v_feats, a_feats), dim=1) 
        return self.classifier(combined)

# ==========================================
# 3. SETUP TRAINING & EARLY STOPPING
# ==========================================
weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_encoded), y=y_encoded)
class_weights_tensor = torch.tensor(weights, dtype=torch.float32)

k_folds = 5
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

all_true_labels = []
all_predictions = []
fold_accuracies = []

MAX_EPOCHS = 100
PATIENCE = 10  

print(f"\nStarting 5-Fold CV (Binary Classification)...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_visual, y_encoded)):
    print(f"\n--- FOLD {fold + 1}/{k_folds} ---")
    
    X_v_train, X_v_val = X_visual[train_idx], X_visual[val_idx]
    X_a_train, X_a_val = X_audio[train_idx], X_audio[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]
    
    scaler_v = StandardScaler()
    X_v_train_scaled = scaler_v.fit_transform(X_v_train)
    X_v_val_scaled = scaler_v.transform(X_v_val)
    
    scaler_a = StandardScaler()
    X_a_train_scaled = scaler_a.fit_transform(X_a_train)
    X_a_val_scaled = scaler_a.transform(X_a_val)
    
    train_loader = DataLoader(TensorDataset(
        torch.tensor(X_v_train_scaled, dtype=torch.float32), 
        torch.tensor(X_a_train_scaled, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long)
    ), batch_size=32, shuffle=True)
    
    val_loader = DataLoader(TensorDataset(
        torch.tensor(X_v_val_scaled, dtype=torch.float32), 
        torch.tensor(X_a_val_scaled, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.long)
    ), batch_size=32, shuffle=False)
    
    model = DeepLateFusionMLP(visual_dim=X_visual.shape[1], audio_dim=X_audio.shape[1], num_classes=len(le.classes_))
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor) 
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, verbose=True
    )
    best_val_loss = float('inf')
    epochs_no_improve = 0
    
    for epoch in range(MAX_EPOCHS):
        model.train()
        for inputs_v, inputs_a, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs_v, inputs_a) 
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs_v, inputs_a, labels in val_loader:
                outputs = model(inputs_v, inputs_a)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
        val_loss /= len(val_loader)
        scheduler.step(val_loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        best_model_state = model.state_dict()  # ← SALVA IL MIGLIOR STATO
    else:
        epochs_no_improve += 1
    
    if epochs_no_improve >= PATIENCE:
        print(f"Early stopping triggered at Epoch {epoch + 1}! (Best Val Loss: {best_val_loss:.4f})")
        break
            
    # RIPRISTINA il miglior modello prima della valutazione
    model.load_state_dict(best_model_state)
    model.eval()
    fold_preds, fold_labels = [], []
    with torch.no_grad():
        for inputs_v, inputs_a, labels in val_loader:
            outputs = model(inputs_v, inputs_a)
            _, preds = torch.max(outputs, 1)
            fold_preds.extend(preds.numpy())
            fold_labels.extend(labels.numpy())
            
    fold_acc = accuracy_score(fold_labels, fold_preds)
    fold_accuracies.append(fold_acc)
    print(f"Fold {fold + 1} Final Accuracy: {fold_acc:.4f}")
    
    all_true_labels.extend(fold_labels)
    all_predictions.extend(fold_preds)

print("\n" + "="*45)
print("=== FINAL BINARY LATE FUSION RESULTS ===")
print("="*45)
print(f"Average Accuracy: {np.mean(fold_accuracies):.4f} (+/- {np.std(fold_accuracies):.4f})\n")
print("Detailed Classification Report:")
print(classification_report(all_true_labels, all_predictions, target_names=le.classes_))

Data Loaded! Binary Distribution:
Non-Positive: 188 videos
Positive: 258 videos

Starting 5-Fold CV (Binary Classification)...

--- FOLD 1/5 ---


c:\Users\alberto.ferrante_kin\Desktop\thesis-notebooks\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Fold 1 Final Accuracy: 0.6333

--- FOLD 2/5 ---


c:\Users\alberto.ferrante_kin\Desktop\thesis-notebooks\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Fold 2 Final Accuracy: 0.6292

--- FOLD 3/5 ---


...

Fold 3 Final Accuracy: 0.5955

--- FOLD 4/5 ---


c:\Users\alberto.ferrante_kin\Desktop\thesis-notebooks\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Fold 4 Final Accuracy: 0.6067

--- FOLD 5/5 ---


c:\Users\alberto.ferrante_kin\Desktop\thesis-notebooks\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Fold 5 Final Accuracy: 0.5955

=== FINAL BINARY LATE FUSION RESULTS ===
Average Accuracy: 0.6121 (+/- 0.0163)

Detailed Classification Report:
              precision    recall  f1-score   support

Non-Positive       0.54      0.49      0.52       188
    Positive       0.65      0.70      0.68       258

    accuracy                           0.61       446
   macro avg       0.60      0.60      0.60       446
weighted avg       0.61      0.61      0.61       446



## Cross Attention in Fusione Intermedia

In [ ]:
import torch
import torch.nn as nn


SHARED_DIM = 128 # 64 o 256 e divisibile per NUM_HEADS
NUM_HEADS = 2 # o 4 o 8
DROPOUT = 0.1 # 0.1 o 0.2 o 0.3
LEARNING_RATE = 3e-4 # 1e-3

# ==========================================
# MODULO 1: Cross-Modal Attention
# ==========================================
class CrossModalAttention(nn.Module):
    """
    Il visual branch usa l'audio come contesto tramite cross-attention.
    - Query  = visual features  (il modello 'chiede' informazioni all'audio)
    - Key/Value = audio features (l'audio 'risponde' con il suo contenuto)
    Il risultato è una versione delle feature visive arricchita dal contesto audio.
    """
    def __init__(self, dim=128, NUM_HEADS=2, DROPOUT=0.1):
        super().__init__()
        assert dim % NUM_HEADS == 0, "dim deve essere divisibile per NUM_HEADS"
        
        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=NUM_HEADS,
            dropout=DROPOUT,
            batch_first=True   # input shape: (batch, seq, dim) — più intuitivo
        )
        self.norm = nn.LayerNorm(dim)
        self.Dropout = nn.Dropout(DROPOUT)



    def forward(self, visual_feats, audio_feats):
        """
        Args:
            visual_feats: (B, dim) — feature visive dopo l'encoder
            audio_feats:  (B, dim) — feature audio dopo l'encoder
        Returns:
            (B, dim) — visual features aggiornate con contesto audio
        """
        # MultiheadAttention si aspetta (B, seq_len, dim)
        # I nostri vettori sono già aggregati nel tempo → seq_len = 1
        q = visual_feats.unsqueeze(1)   # (B, 1, dim)
        k = audio_feats.unsqueeze(1)    # (B, 1, dim)
        v = audio_feats.unsqueeze(1)    # (B, 1, dim)

        attended, _ = self.attn(q, k, v)  # (B, 1, dim)
        attended = attended.squeeze(1)     # (B, dim)

        # Connessione residuale + normalizzazione
        out = self.norm(visual_feats + self.Dropout(attended))
        return out  # (B, dim)


# ==========================================
# MODULO 2: Rete con CrossModalAttention integrata
# ==========================================
class FusionWithCrossAttention(nn.Module):
    def __init__(self, visual_dim=512, audio_dim=128, SHARED_DIM=128, num_classes=3, NUM_HEADS=2, DROPOUT=0.1):
        super().__init__()

        # Encoder separati: proiettano entrambe le modalità in SHARED_DIM 
        self.visual_encoder = nn.Sequential(
            nn.Linear(visual_dim, 256),
            nn.LayerNorm(256), nn.GELU(), nn.Dropout(DROPOUT),
            #nn.Linear(512, 256),               # ← layer aggiuntivo
            nn.LayerNorm(256), nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(256, SHARED_DIM),
            nn.LayerNorm(SHARED_DIM), nn.GELU()
        )
        self.audio_encoder = nn.Sequential(
            nn.Linear(audio_dim, 256),
            nn.LayerNorm(256), nn.GELU(), nn.Dropout(DROPOUT),
            #nn.Linear(512, 256),               # ← layer aggiuntivo
            nn.LayerNorm(256), nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(256, SHARED_DIM),
            nn.LayerNorm(SHARED_DIM), nn.GELU()
        )

        # Cross-attention: visual guarda l'audio
        self.cross_attn = CrossModalAttention(dim=SHARED_DIM , NUM_HEADS=NUM_HEADS)
        self.cross_attn_a = CrossModalAttention(dim=SHARED_DIM , NUM_HEADS=NUM_HEADS)

        # Classifier sulla concatenazione (visual_attended || audio_enc)
        self.classifier = nn.Sequential(
            nn.Linear(SHARED_DIM  * 2, 128),
            nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, visual_x, audio_x):
        v = self.visual_encoder(visual_x)   # (B, SHARED_DIM )
        a = self.audio_encoder(audio_x)     # (B, SHARED_DIM )

        # Visual arricchito dal contesto audio
        v_attended = self.cross_attn(v, a)  # (B, SHARED_DIM )
        a_attended = self.cross_attn_a(a, v)   # audio guarda visual

        combined = torch.cat([v_attended, a_attended], dim=1)

        # Fusione finale
        #combined = torch.cat([v_attended, a], dim=1)  # (B, SHARED_DIM  * 2)
        return self.classifier(combined)
    

# ==========================================
# 3. SETUP TRAINING & EARLY STOPPING
# ==========================================
weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_encoded), y=y_encoded)
class_weights_tensor = torch.tensor(weights, dtype=torch.float32)

k_folds = 5
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

all_true_labels = []
all_predictions = []
fold_accuracies = []

MAX_EPOCHS = 100
PATIENCE = 10  




In [50]:
import logging
import sys

logger = logging.getLogger()
logger.setLevel(logging.INFO)

file_handler = logging.FileHandler("training_log.txt", mode="w")
file_handler.setLevel(logging.INFO)

console_handler = logging.StreamHandler(sys.stdout)
console_handler.setLevel(logging.INFO)

formatter = logging.Formatter("%(asctime)s - %(message)s")
file_handler.setFormatter(formatter)
console_handler.setFormatter(formatter)

logger.handlers = []  # evita duplicati
logger.addHandler(file_handler)
logger.addHandler(console_handler)

def log(msg):
    logger.info(msg)

results = []


#SHARED_DIM': 256,
#  'NUM_HEADS': 2,
#  'DROPOUT': 0.2,
#  'LR': 0.0001,

SHARED_DIMS = [256]
NUM_HEADS_LIST = [2, 4, 8]
DROPOUTS = [0.2, 0.3, 0.4]
LEARNING_RATES = [0.001, 0.0001, 0.00001]

for SHARED_DIM in SHARED_DIMS:
    for NUM_HEADS in NUM_HEADS_LIST:
        for LEARNING_RATE in LEARNING_RATES:
            for DROPOUT in DROPOUTS:
            
                #log("\n" + "="*60)
                #log(f"TEST CONFIG:")
                #log(f"SHARED_DIM={SHARED_DIM}, NUM_HEADS={NUM_HEADS}, DROPOUT={DROPOUT}, LR={LEARNING_RATE}")
                #log("="*60)

                fold_accuracies = []

                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

                for fold, (train_idx, val_idx) in enumerate(skf.split(X_visual, y_encoded)):

                    #print(f"\n--- FOLD {fold + 1}/5 ---")

                    # ====== QUI RESTA TUTTO IDENTICO AL TUO CODICE ======
                    X_v_train, X_v_val = X_visual[train_idx], X_visual[val_idx]
                    X_a_train, X_a_val = X_audio[train_idx], X_audio[val_idx]
                    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]

                    scaler_v = StandardScaler()
                    X_v_train_scaled = scaler_v.fit_transform(X_v_train)
                    X_v_val_scaled = scaler_v.transform(X_v_val)

                    scaler_a = StandardScaler()
                    X_a_train_scaled = scaler_a.fit_transform(X_a_train)
                    X_a_val_scaled = scaler_a.transform(X_a_val)

                    train_loader = DataLoader(TensorDataset(
                        torch.tensor(X_v_train_scaled, dtype=torch.float32),
                        torch.tensor(X_a_train_scaled, dtype=torch.float32),
                        torch.tensor(y_train, dtype=torch.long)
                    ), batch_size=32, shuffle=True)

                    val_loader = DataLoader(TensorDataset(
                        torch.tensor(X_v_val_scaled, dtype=torch.float32),
                        torch.tensor(X_a_val_scaled, dtype=torch.float32),
                        torch.tensor(y_val, dtype=torch.long)
                    ), batch_size=32, shuffle=False)

                    model = FusionWithCrossAttention(
                        visual_dim=X_visual.shape[1],
                        audio_dim=X_audio.shape[1],
                        SHARED_DIM=SHARED_DIM,
                        num_classes=len(le.classes_),
                        NUM_HEADS=NUM_HEADS,
                        DROPOUT=DROPOUT
                    )

                    class_weights_tensor = torch.tensor(
                        compute_class_weight(
                            class_weight='balanced',
                            classes=np.unique(y_encoded),
                            y=y_encoded
                        ),
                        dtype=torch.float32
                    )

                    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
                    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

                    #scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                    #    optimizer, mode='min', factor=0.5, patience=5
                    #)

                    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                        optimizer, mode='min', factor=0.5, patience=PATIENCE  # era 5
                    )

                    best_val_loss = float('inf')
                    epochs_no_improve = 0

                    for epoch in range(MAX_EPOCHS):
                        model.train()

                        for inputs_v, inputs_a, labels in train_loader:
                            optimizer.zero_grad()
                            outputs = model(inputs_v, inputs_a)
                            loss = criterion(outputs, labels)
                            loss.backward()
                            optimizer.step()

                        model.eval()
                        val_loss = 0.0

                        with torch.no_grad():
                            for inputs_v, inputs_a, labels in val_loader:
                                outputs = model(inputs_v, inputs_a)
                                loss = criterion(outputs, labels)
                                val_loss += loss.item()

                        val_loss /= len(val_loader)
                        scheduler.step(val_loss)

                        if val_loss < best_val_loss:
                            best_val_loss = val_loss
                            epochs_no_improve = 0
                            best_model_state = model.state_dict()
                        else:
                            epochs_no_improve += 1

                        if epochs_no_improve >= PATIENCE:
                            break

                    model.load_state_dict(best_model_state)
                    model.eval()

                    preds, labels_list = [], []

                    with torch.no_grad():
                        for inputs_v, inputs_a, labels in val_loader:
                            outputs = model(inputs_v, inputs_a)
                            _, p = torch.max(outputs, 1)
                            preds.extend(p.numpy())
                            labels_list.extend(labels.numpy())

                    acc = accuracy_score(labels_list, preds)
                    fold_accuracies.append(acc)

                mean_acc = np.mean(fold_accuracies)

                results.append({
                    "SHARED_DIM": SHARED_DIM,
                    "NUM_HEADS": NUM_HEADS,
                    "DROPOUT": DROPOUT,
                    "LR": LEARNING_RATE,
                    "ACC": mean_acc
                })
                log(f"Tested Config: SHARED_DIM={SHARED_DIM}, NUM_HEADS={NUM_HEADS}, DROPOUT={DROPOUT}, LR={LEARNING_RATE} -> MEAN ACC: {mean_acc:.4f}")
                #log(f"\n>>> MEAN ACC: {mean_acc:.4f}")
                #log(f"Fold {fold+1} Epoch {epoch} Loss {val_loss:.4f}")



2026-04-28 18:09:49,996 - Tested Config: SHARED_DIM=256, NUM_HEADS=2, DROPOUT=0.2, LR=0.001 -> MEAN ACC: 0.6189
2026-04-28 18:09:59,614 - Tested Config: SHARED_DIM=256, NUM_HEADS=2, DROPOUT=0.3, LR=0.001 -> MEAN ACC: 0.5942
2026-04-28 18:10:09,047 - Tested Config: SHARED_DIM=256, NUM_HEADS=2, DROPOUT=0.4, LR=0.001 -> MEAN ACC: 0.6031


KeyboardInterrupt: 

In [39]:
#max(results, key=lambda x: x['ACC'])
sorted(results, key=lambda x: x["ACC"], reverse=True)[:5]

[{'SHARED_DIM': 256,
  'NUM_HEADS': 2,
  'DROPOUT': 0.1,
  'LR': 0.001,
  'ACC': np.float64(0.6323595505617977)},
 {'SHARED_DIM': 256,
  'NUM_HEADS': 2,
  'DROPOUT': 0.3,
  'LR': 0.0001,
  'ACC': np.float64(0.6279900124843945)},
 {'SHARED_DIM': 256,
  'NUM_HEADS': 2,
  'DROPOUT': 0.4,
  'LR': 0.001,
  'ACC': np.float64(0.6255181023720349)},
 {'SHARED_DIM': 256,
  'NUM_HEADS': 8,
  'DROPOUT': 0.2,
  'LR': 1e-05,
  'ACC': np.float64(0.623370786516854)},
 {'SHARED_DIM': 256,
  'NUM_HEADS': 4,
  'DROPOUT': 0.1,
  'LR': 0.001,
  'ACC': np.float64(0.6210986267166042)}]

###### {'SHARED_DIM': 256,
######   'NUM_HEADS': 2,
######   'DROPOUT': 0.2,
######   'LR': 0.0001,
######   'ACC': np.float64(0.6436204744069913)}